# 🛡️ Cadence Financial Group — Enterprise PII Sanitization Pipeline
### Case Study 2: Text Analysis & PII Detection | Team Integration & Master Demo Notebook

**Objective:** Ingest confidential multi-format documents (`.docx`, `.pptx`, `.pdf`), extract structure and text, detect sensitive Personally Identifiable Information (PII) using a hybrid Regex + NER engine, apply offset-safe right-to-left redaction, and enforce corporate data governance via the **Privacy Safety Gate** before any data can be passed downstream.

---
### 👥 Team Roles & Responsibilities
* **Person 1 (Extraction Lead):** Native DOCX/PPTX parsing, PyMuPDF extraction, and Tesseract OCR fallback.
* **Person 2 (PII Detection Lead):** High-precision Regex (Emails, Phones, IPs, IDs) + spaCy NER (Person Names).
* **Person 3 (Redaction & QA Lead):** Offset-safe redaction logic, findings audit log, and ground-truth validation.
* **Person 4 (Integration & Demo Owner):** Master Colab/Jupyter notebook, sequential pipeline orchestration, and safety gate.
* **Person 5 (Architecture & Presentation Lead):** Solution blueprint, technical architecture, and stakeholder deck.

> **🔒 Privacy Mandate:** 100% Local processing. Raw document text is strictly quarantined and NEVER transmitted to third-party LLMs or cloud APIs.

## 1. Environment Setup & Dependency Initialization
Install required local libraries and load the lightweight spaCy language model.

In [ ]:
# Install dependencies (if running in Google Colab)
# !pip install -q python-docx python-pptx pymupdf spacy pytesseract
# !python -m spacy download en_core_web_sm

import os
import sys
import re
import json
import csv
import time
from pathlib import Path
from collections import Counter
import pandas as pd

# Ensure repo root is accessible
REPO_ROOT = Path(".").resolve()
if (REPO_ROOT / "extractor.py").exists():
    sys.path.insert(0, str(REPO_ROOT))
elif (REPO_ROOT.parent / "extractor.py").exists():
    sys.path.insert(0, str(REPO_ROOT.parent))

from extractor import extract_file
from pii_detector import detect_pii

print("✅ Setup completed: Extractor & PII Detector successfully initialized locally.")


### 📋 Cross-Team Technical Contracts (Data Schemas)
To allow independent work streams, the pipeline strictly complies with the agreed schemas:
* **Schema A (Extractor Output):** `file_name`, `file_type`, `segment_id`, `location_type`, `location_index`, `text`, `extraction_method`, `warning`
* **Schema B (Detector Output):** `segment_id`, `entity_text`, `entity_type`, `start_char`, `end_char`, `detection_method`, `confidence_or_strength`

Let's verify this contract against the canonical benchmark segment:

In [ ]:
test_text = "Contact Marguerite Vossberg at m.vossberg@cadencefg.example"
test_hits = detect_pii(test_text, segment_id="Cadence_Org.pptx#00012")
print("✅ Conforming Schema B Output:")
print(json.dumps(test_hits[0], indent=2))


## 2. Document Ingestion & Validation
Validate the three enterprise artifacts provided for the Cadence case study.

In [ ]:
TARGET_FILES = [
    "Cadence_TPRM_Training.docx",
    "Cadence_Group_Risk_Management_Policy__v6.0.pdf",
    "Cadence_Financial_Group_Organizational_Pack.pptx"
]

ingested_paths = []
search_dirs = [Path("."), Path(".."), Path("data"), Path("../data")]

for filename in TARGET_FILES:
    found = False
    for sdir in search_dirs:
        candidate = sdir / filename
        if candidate.exists():
            ingested_paths.append(candidate.resolve())
            size_kb = candidate.stat().st_size / 1024
            print(f"✅ Ingested: {filename} ({size_kb:,.1f} KB)")
            found = True
            break
    if not found:
        print(f"⚠️ Warning: {filename} not found in search paths.")

## 3. Structural Text Extraction (Person 1 Module)
Extract paragraphs, slides, and PDF pages. Scanned pages invoke OCR fallback.

In [ ]:
all_document_segments = {}

# Check for pre-cached extractions if OCR binary is unavailable in local environment
cache_path = Path("person1_sahib/extraction_output.json")
cached_records = {}
if cache_path.exists():
    with open(cache_path, "r", encoding="utf-8") as cf:
        for item in json.load(cf):
            cached_records.setdefault(item["file_name"], []).append(item)

for fpath in ingested_paths:
    fn = fpath.name
    print(f"Extracting {fn}...")
    try:
        res = extract_file(fpath)
        records = res.to_records()
        # Check if PDF required OCR and had binary missing
        if any("tesseract" in str(f).lower() for f in getattr(res, "failures", [])) and fn in cached_records:
            print(f"  -> Using validated extraction cache for scanned PDF {fn}")
            records = cached_records[fn]
    except Exception as e:
        if fn in cached_records:
            records = cached_records[fn]
        else:
            records = []
    all_document_segments[fn] = records
    print(f"  Extracted {len(records)} segments.")

## 4. Hybrid PII Detection Engine (Person 2 Module)
Scan each segment using high-precision Regex and spaCy Named Entity Recognition (NER).

In [ ]:
all_findings = []
finding_id = 0

for fn, segments in all_document_segments.items():
    for seg in segments:
        text = seg.get("text", "")
        if not text:
            continue
        
        seg_id = seg.get("segment_id", "unknown")
        hits = detect_pii(text, segment_id=seg_id)
        
        for hit in hits:
            finding_id += 1
            item = dict(hit)
            item["finding_id"] = f"PII-{finding_id:05d}"
            item["file_name"] = fn
            item["location_type"] = seg.get("location_type", "")
            item["location_index"] = seg.get("location_index", "")
            item["page_or_slide"] = seg.get("page_or_slide", "")
            item["segment_warning"] = seg.get("warning", "")
            
            # Mask for safe display
            val = hit.get("entity_text", "")
            etype = hit.get("entity_type", "")
            if etype == "EMAIL" and "@" in val:
                p = val.split("@")
                item["masked_value"] = f"{p[0][:1]}***@{p[1]}"
            elif etype == "PHONE":
                item["masked_value"] = val[:4] + " *** " + val[-3:] if len(val) > 6 else "***"
            elif etype == "PERSON_NAME":
                item["masked_value"] = " ".join(w[0] + "***" if len(w) > 1 else w for w in val.split())
            else:
                item["masked_value"] = val[:2] + "***" if len(val) > 3 else "***"
                
            all_findings.append(item)

df_findings = pd.DataFrame(all_findings)
print(f"🎯 Total PII Entities Detected: {len(df_findings)}")
print("\nBreakdown by Category:")
print(df_findings["entity_type"].value_counts())

## 5. Offset-Safe (Right-to-Left) Redaction Engine
Replace detected spans with transparent tags (e.g., `[REDACTED: EMAIL]`) strictly sorted from right to left to prevent index drift.

In [ ]:
def redact_text_rtl(text: str, findings: list) -> str:
    if not text or not findings:
        return text
    # Sort by start_char descending (right-to-left)
    valid = [f for f in findings if 0 <= f.get("start_char", -1) < f.get("end_char", -1) <= len(text)]
    valid.sort(key=lambda x: (x["start_char"], x["end_char"]), reverse=True)
    
    sanitized = text
    last_start = len(text) + 1
    for f in valid:
        if f["end_char"] <= last_start:
            s, e = f["start_char"], f["end_char"]
            tag = f"[REDACTED: {f.get('entity_type', 'PII')}]"
            sanitized = sanitized[:s] + tag + sanitized[e:]
            last_start = s
    return sanitized

# Map findings by segment
findings_by_seg = {}
for f in all_findings:
    findings_by_seg.setdefault(f["segment_id"], []).append(f)

sanitized_documents = {}
for fn, segments in all_document_segments.items():
    sanitized_parts = []
    for seg in segments:
        raw = seg.get("text", "")
        seg_findings = findings_by_seg.get(seg.get("segment_id"), [])
        redacted = redact_text_rtl(raw, seg_findings)
        seg["redacted_text"] = redacted
        if redacted:
            sanitized_parts.append(redacted)
    sanitized_documents[fn] = "\n\n".join(sanitized_parts)

print("🛡️ Redaction completed across all documents.")

## 6. Live Before vs. After Redaction Inspection

In [ ]:
print("=== LIVE REDACTION SAMPLES ===\n")
shown = 0
for fn, segments in all_document_segments.items():
    for seg in segments:
        raw = seg.get("text", "")
        red = seg.get("redacted_text", "")
        if raw != red and "[REDACTED:" in red:
            shown += 1
            print(f"Sample {shown} [{fn} - {seg.get('location_type')} {seg.get('location_index')}]:")
            print(f"  RAW      : {raw[:120]}...")
            print(f"  REDACTED : {red[:120]}...")
            print("-" * 70)
            if shown >= 3:
                break
    if shown >= 3:
        break

## 7. Cadence Privacy Safety Gate & Governance Audit
Evaluate whether any documents contained OCR errors, low-confidence passages, or unreadable pages. Documents with warnings receive a mandatory `REVIEW REQUIRED` status.

In [ ]:
print("=== PRIVACY SAFETY GATE AUDIT ===\n")
gate_results = []
for fn, segments in all_document_segments.items():
    warns = [s for s in segments if s.get("warning")]
    is_clean = len(warns) == 0
    verdict = "PASS - AUDITED" if is_clean else "REVIEW REQUIRED"
    gate_results.append({
        "Document": fn,
        "Total Segments": len(segments),
        "Warnings Logged": len(warns),
        "Safety Gate Status": verdict,
        "Safe for Downstream AI": is_clean
    })

df_gate = pd.DataFrame(gate_results)
print(df_gate.to_string(index=False))

## 8. Export Sanitized Artifacts & Audit Logs
Save the sanitized outputs, `findings.csv`, `findings.json`, and `qa_log.csv` to `04_Results/`.

In [ ]:
results_dir = Path("04_Results")
sanitized_dir = results_dir / "sanitized_outputs"
sanitized_dir.mkdir(parents=True, exist_ok=True)

# Export sanitized texts
for fn, text in sanitized_documents.items():
    stem = Path(fn).stem
    out_path = sanitized_dir / f"{stem}_sanitized.txt"
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(text)
    print(f"Saved sanitized text: {out_path}")

# Export findings.csv and findings.json
df_findings.to_csv(results_dir / "findings.csv", index=False, encoding="utf-8")
with open(results_dir / "findings.json", "w", encoding="utf-8") as f:
    json.dump(all_findings, f, indent=2, ensure_ascii=False)

print(f"\n✅ Export complete! All artifacts preserved in {results_dir.resolve()}.")